# Lab 4 - Observation Tasks

## Observation Task 1 - Using Uniform Cost Search

In [54]:
# Weighted adjacency list: node -> list of (neighbor, cost)
romania_weighted = {
    'Arad': [('Zerind', 75), ('Sibiu', 140), ('Timisoara', 118)],
    'Zerind': [('Arad', 75), ('Oradea', 71)],
    'Oradea': [('Zerind', 71), ('Sibiu', 151)],
    'Sibiu': [('Arad', 140), ('Oradea', 151), ('Fagaras', 99), ('Rimnicu Vilcea', 80)],
    'Timisoara': [('Arad', 118), ('Lugoj', 111)],
    'Lugoj': [('Timisoara', 111), ('Mehadia', 70)],
    'Mehadia': [('Lugoj', 70), ('Drobeta', 75)],
    'Drobeta': [('Mehadia', 75), ('Craiova', 120)],
    'Craiova': [('Drobeta', 120), ('Rimnicu Vilcea', 146), ('Pitesti', 138)],
    'Rimnicu Vilcea': [('Sibiu', 80), ('Craiova', 146), ('Pitesti', 97)],
    'Fagaras': [('Sibiu', 99), ('Bucharest', 211)],
    'Pitesti': [('Rimnicu Vilcea', 97), ('Craiova', 138), ('Bucharest', 101)],
    'Bucharest': [('Fagaras', 211), ('Pitesti', 101), ('Giurgiu', 90), ('Urziceni', 85)],
    'Giurgiu': [('Bucharest', 90)],
    'Urziceni': [('Bucharest', 85), ('Hirsova', 98), ('Vaslui', 142)],
    'Hirsova': [('Urziceni', 98), ('Eforie', 86)],
    'Eforie': [('Hirsova', 86)],
    'Vaslui': [('Urziceni', 142), ('Iasi', 92)],
    'Iasi': [('Vaslui', 92), ('Neamt', 87)],
    'Neamt': [('Iasi', 87)]
}

START = 'Arad'
GOAL = 'Bucharest'

In [55]:
import heapq

def uniform_cost_search(graph, start, goal):
    # Priority queue entries: (cumulative_cost, current_node, path)
    frontier = [(0, start, [start])]
    explored = set()
    nodes_expanded = 0

    while frontier:
        cost, node, path = heapq.heappop(frontier)

        if node == goal:
            return path, cost, nodes_expanded

        if node in explored:
            continue

        explored.add(node)
        nodes_expanded += 1

        for neighbor, edge_cost in graph[node]:
            if neighbor not in explored:
                heapq.heappush(
                    frontier,
                    (cost + edge_cost, neighbor, path + [neighbor])
                )

    return None, float('inf'), nodes_expanded


path, cost, expanded = uniform_cost_search(
    romania_weighted, START, GOAL
)

print("Path:", path)
print("Total Cost:", cost)
print("Nodes Expanded:", expanded)

Path: ['Arad', 'Sibiu', 'Rimnicu Vilcea', 'Pitesti', 'Bucharest']
Total Cost: 418
Nodes Expanded: 12


## Observation Task 2 - Using Depth-Limited Search

In [56]:
def depth_limited_search(graph, node, goal, limit, path, visited, counter):
    counter[0] += 1

    if node == goal:
        return path

    if limit == 0:
        return 'CUTOFF'

    cutoff_occurred = False

    for neighbor, _ in graph[node]:
        if neighbor not in visited:
            visited.add(neighbor)

            result = depth_limited_search(
                graph, neighbor, goal, limit - 1,
                path + [neighbor], visited, counter
            )

            visited.remove(neighbor)

            if result == 'CUTOFF':
                cutoff_occurred = True
            elif result != 'FAILURE':
                return result

    return 'CUTOFF' if cutoff_occurred else 'FAILURE'


def DLS(graph, start, goal, limit):
    counter = [0]

    result = depth_limited_search(
        graph, start, goal, limit, [start], {start}, counter
    )

    return result, counter[0]


result, expanded = DLS(romania_weighted, START, GOAL, 3)

print("Using Depth-Limited Search")
print("Result:", result)
print("Nodes Expanded:", expanded)

Using Depth-Limited Search
Result: ['Arad', 'Sibiu', 'Fagaras', 'Bucharest']
Nodes Expanded: 9


## Observation Task 3 - Using Iterative Deepening DFS

In [57]:
def iterative_deepening_dfs(graph, start, goal, max_depth=20):
    total_expanded = 0

    for depth in range(max_depth + 1):
        counter = [0]

        result = depth_limited_search(
            graph, start, goal, depth, [start], {start}, counter
        )

        total_expanded += counter[0]

        if result not in ('CUTOFF', 'FAILURE'):
            return result, depth, total_expanded

        if result == 'FAILURE':
            break

    return None, -1, total_expanded


result, depth, expanded = iterative_deepening_dfs(
    romania_weighted, START, GOAL
)

print("Using Iterative Deepening DFS")
print("Path:", result)
print("Depth:", depth)
print("Total Nodes Expanded:", expanded)

Using Iterative Deepening DFS
Path: ['Arad', 'Sibiu', 'Fagaras', 'Bucharest']
Depth: 3
Total Nodes Expanded: 23


## Observation Task 4 - Run All Together

In [58]:
def run_all():
    print("=" * 60)
    print(f"Start: {START} | Goal: {GOAL}")
    print("=" * 60)

    # ---- UCS ----
    path, cost, expanded = uniform_cost_search(
        romania_weighted, START, GOAL
    )

    print("\n[Uniform Cost Search]")
    print(" Path :", " -> ".join(path))
    print(" Cost :", cost, "km")
    print(" Depth :", len(path) - 1)
    print(" Expanded :", expanded, "nodes")

    # ---- DLS with user-defined limit ----
    limit = 3
    path, expanded = DLS(romania_weighted, START, GOAL, limit)

    print(f"\n[Depth-Limited Search | limit = {limit}]")

    if path in ('CUTOFF', 'FAILURE'):
        print(" Result :", path)
    else:
        print(" Path :", " -> ".join(path))

    print(" Expanded :", expanded, "nodes")

    # ---- IDDFS ----
    path, depth, expanded = iterative_deepening_dfs(
        romania_weighted, START, GOAL
    )

    print("\n[Iterative Deepening DFS]")

    if path:
        print(" Path :", " -> ".join(path))
        print(" Depth :", depth)
    else:
        print(" Result : FAILURE")

    print(" Expanded :", expanded, "nodes (cumulative)")


if __name__ == "__main__":
    run_all()

Start: Arad | Goal: Bucharest

[Uniform Cost Search]
 Path : Arad -> Sibiu -> Rimnicu Vilcea -> Pitesti -> Bucharest
 Cost : 418 km
 Depth : 4
 Expanded : 12 nodes

[Depth-Limited Search | limit = 3]
 Path : Arad -> Sibiu -> Fagaras -> Bucharest
 Expanded : 9 nodes

[Iterative Deepening DFS]
 Path : Arad -> Sibiu -> Fagaras -> Bucharest
 Depth : 3
 Expanded : 23 nodes (cumulative)


## Observation Task 5 - Extend BFS for Weighted Comparison 

In [59]:
from collections import deque

def bfs_weighted(graph, start, goal):
    """BFS ignoring weights — used for comparison."""
    frontier = deque([[start]])
    explored = set()
    expanded = 0

    while frontier:
        path = frontier.popleft()
        node = path[-1]

        if node in explored:
            continue

        explored.add(node)
        expanded += 1

        if node == goal:
            return path, expanded

        for neighbor, _ in graph[node]:
            if neighbor not in explored:
                frontier.append(path + [neighbor])

    return None, expanded


# Run BFS
path, expanded = bfs_weighted(romania_weighted, START, GOAL)

print("\nUsing Breadth-First Search (BFS)")

if path:
    print("Path:", " -> ".join(path))
    print("Depth:", len(path) - 1)
else:
    print("Result: Goal not found")

print("Expanded:", expanded, "nodes")


Using Breadth-First Search (BFS)
Path: Arad -> Sibiu -> Fagaras -> Bucharest
Depth: 3
Expanded: 9 nodes


## Implementation Task 1 - Verify That the Adjacency List is Symmetric

In [60]:
def check_symmetry(graph):
    for city in graph:
        for neighbor, cost in graph[city]:
            reverse_edges = graph.get(neighbor, [])

            if (city, cost) not in reverse_edges:
                print(f"Mismatch: {city} -> {neighbor}, cost = {cost}")
                return

    print("All edges have matching costs in both directions.")


check_symmetry(romania_weighted)

All edges have matching costs in both directions.


## Implementation Task 2.1 - Modify UCS to return the complete frontier at each expansion step (print for  debugging).

In [61]:
import heapq

def uniform_cost_search(graph, start, goal):
    frontier = [(0, start, [start])]
    explored = set()
    nodes_expanded = 0
    frontier_history = []

    print("\n[Uniform Cost Search]")
    print("Initial Frontier:", frontier)

    while frontier:
        cost, node, path = heapq.heappop(frontier)

        if node in explored:
            continue

        if node == goal:
            print("Goal Found:", node)
            print("Path:", " -> ".join(path))
            print("Total Cost:", cost, "km")
            return path, cost, nodes_expanded, frontier_history

        explored.add(node)
        nodes_expanded += 1

        for neighbor, edge_cost in graph[node]:
            if neighbor not in explored:
                heapq.heappush(
                    frontier,
                    (cost + edge_cost, neighbor, path + [neighbor])
                )

        current_frontier = sorted(frontier)
        frontier_history.append({
            "expanded": node,
            "frontier": current_frontier.copy()
        })

        print(f"\nAfter expanding {node}:")
        for item in current_frontier:
            print(item)

    return None, float('inf'), nodes_expanded, frontier_history


path, cost, expanded, frontier_history = uniform_cost_search(
    romania_weighted, START, GOAL
)

print("\nTotal Nodes Expanded:", expanded)
print("Frontier snapshots saved:", len(frontier_history))


[Uniform Cost Search]
Initial Frontier: [(0, 'Arad', ['Arad'])]

After expanding Arad:
(75, 'Zerind', ['Arad', 'Zerind'])
(118, 'Timisoara', ['Arad', 'Timisoara'])
(140, 'Sibiu', ['Arad', 'Sibiu'])

After expanding Zerind:
(118, 'Timisoara', ['Arad', 'Timisoara'])
(140, 'Sibiu', ['Arad', 'Sibiu'])
(146, 'Oradea', ['Arad', 'Zerind', 'Oradea'])

After expanding Timisoara:
(140, 'Sibiu', ['Arad', 'Sibiu'])
(146, 'Oradea', ['Arad', 'Zerind', 'Oradea'])
(229, 'Lugoj', ['Arad', 'Timisoara', 'Lugoj'])

After expanding Sibiu:
(146, 'Oradea', ['Arad', 'Zerind', 'Oradea'])
(220, 'Rimnicu Vilcea', ['Arad', 'Sibiu', 'Rimnicu Vilcea'])
(229, 'Lugoj', ['Arad', 'Timisoara', 'Lugoj'])
(239, 'Fagaras', ['Arad', 'Sibiu', 'Fagaras'])
(291, 'Oradea', ['Arad', 'Sibiu', 'Oradea'])

After expanding Oradea:
(220, 'Rimnicu Vilcea', ['Arad', 'Sibiu', 'Rimnicu Vilcea'])
(229, 'Lugoj', ['Arad', 'Timisoara', 'Lugoj'])
(239, 'Fagaras', ['Arad', 'Sibiu', 'Fagaras'])
(291, 'Oradea', ['Arad', 'Sibiu', 'Oradea'])

Aft

## Implementation Task 2.2 - DLS: test limits 3, 4 and 5

In [62]:
print("\n[Depth-Limited Search Experiments]")

for limit in [3, 4, 5]:
    path, expanded = DLS(
        romania_weighted, START, GOAL, limit
    )

    print(f"\nDepth Limit: {limit}")

    if path == 'CUTOFF':
        print("Result: CUTOFF")
    elif path == 'FAILURE':
        print("Result: FAILURE")
    else:
        print("Path:", " -> ".join(path))
        print("Solution found at depth:", len(path) - 1)

    print("Nodes Expanded:", expanded)


[Depth-Limited Search Experiments]

Depth Limit: 3
Path: Arad -> Sibiu -> Fagaras -> Bucharest
Solution found at depth: 3
Nodes Expanded: 9

Depth Limit: 4
Path: Arad -> Sibiu -> Fagaras -> Bucharest
Solution found at depth: 3
Nodes Expanded: 11

Depth Limit: 5
Path: Arad -> Zerind -> Oradea -> Sibiu -> Fagaras -> Bucharest
Solution found at depth: 5
Nodes Expanded: 6


## Implementation Task 2.3 - IDDFS: print the result at every depth

In [63]:
def iterative_deepening_dfs(graph, start, goal, max_depth=20):
    total_expanded = 0

    print("\n[Iterative Deepening DFS]")

    for depth in range(max_depth + 1):
        counter = [0]

        result = depth_limited_search(
            graph, start, goal, depth,
            [start], {start}, counter
        )

        total_expanded += counter[0]

        print(f"\nDepth Limit: {depth}")

        if result == 'CUTOFF':
            print("Result: CUTOFF")
        elif result == 'FAILURE':
            print("Result: FAILURE")
        else:
            print("Path:", " -> ".join(result))
            print("Solution Depth:", len(result) - 1)
            print("Total Nodes Expanded:", total_expanded)
            return result, depth, total_expanded

        print("Nodes Expanded This Iteration:", counter[0])

    return None, -1, total_expanded


path, depth, expanded = iterative_deepening_dfs(
    romania_weighted, START, GOAL
)

print("\n[IDDFS Final Result]")
print("Path:", " -> ".join(path) if path else "Not Found")
print("Depth:", depth)
print("Total Nodes Expanded:", expanded)


[Iterative Deepening DFS]

Depth Limit: 0
Result: CUTOFF
Nodes Expanded This Iteration: 1

Depth Limit: 1
Result: CUTOFF
Nodes Expanded This Iteration: 4

Depth Limit: 2
Result: CUTOFF
Nodes Expanded This Iteration: 9

Depth Limit: 3
Path: Arad -> Sibiu -> Fagaras -> Bucharest
Solution Depth: 3
Total Nodes Expanded: 23

[IDDFS Final Result]
Path: Arad -> Sibiu -> Fagaras -> Bucharest
Depth: 3
Total Nodes Expanded: 23
